In [ ]:
import scanpy as sc
import anndata as ad
import scvi
import matplotlib.pyplot as plt 
import sys
sys.path.append("/multiHIVE/src")
from multiHIVE.model import multiHIVE
import pandas as pd
import numpy as np

In [ ]:
adata = sc.read('/Data/combined_berk_bio.h5ad')
adata

In [ ]:
# sc.pp.filter_cells(adata, min_genes=200)
# sc.pp.filter_genes(adata, min_cells=3)
# sc.pp.highly_variable_genes(
#     adata,
#     n_top_genes=2000,
#     flavor="seurat_v3",
#     batch_key="batch",
#     subset=True,
#     layer="counts"
# )

print(adata)
multiHIVE.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    protein_expression_obsm_key="protein_expression"
)
# vae = HierarVI.load("./outputs/saved_model/")
vae = multiHIVE(adata, latent_distribution="normal", n_genes = adata.shape[1], n_proteins = adata.obsm['protein_expression'].shape[1], n_regions = 0)
vae.train()
vae.get_latent_representation()

In [ ]:
vae.save("./outputs/saved_model/", save_anndata=True)

In [ ]:
hvg = 5125

In [ ]:
generated_data = vae.posterior_predictive_sample(adata, swap_latent=False)
rna_sample = generated_data[:,:hvg].copy()
proteins_sample = pd.DataFrame(generated_data[:,hvg:],  index= adata.obs_names, columns = adata.obsm['protein_expression'].columns)
adata.obsm['RNA_Z1_denoised'] = rna_sample
adata.obsm['protein_Z1_denoised'] = proteins_sample

generated_data = vae.posterior_predictive_sample(adata, swap_latent=True)
rna_sample = generated_data[:,:hvg].copy()
proteins_sample = pd.DataFrame(generated_data[:,hvg:],  index= adata.obs_names, columns = adata.obsm['protein_expression'].columns)
adata.obsm['RNA_Z2_denoised'] = rna_sample
adata.obsm['protein_Z2_denoised'] = proteins_sample

In [ ]:
# adata.obsm['RNA_Z2_denoised'] = adata.obsm['RNA_Z2_denoised'].values
# adata.obsm['RNA_Z1_denoised'] = adata.obsm['RNA_Z1_denoised'].values
adata.write("./outputs/Thymus_HierarVi.h5ad")

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
rc_parms = {"figure.figsize": [5, 5], "figure.dpi": 200, "font.size": 10, "font.family": "Arial"}
save_parms = {"bbox_inches": "tight", "transparent": True}
# adata = sc.read_h5ad("./outputs/Thymus_HierarVi.h5ad")
adata
with plt.rc_context(rc_parms):
      ax = sc.pl.umap(adata, color = "annotations", frameon = False, return_fig = True, title = '')
      plt.savefig("./outputs/Zc_umap.png", **save_parms)